<a href="https://colab.research.google.com/github/afizs/python-notes/blob/main/ai-apps/haystack_tonic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PII detection and extraction in Haystack pipelines

## Installation

In [11]:
!pip install textual-haystack -q

## Setup
First thing first get your textual FREE API Key.
https://docs.tonic.ai/textual/tonic-textual-getting-started

In [12]:
import os
from google.colab import userdata

# Load the TONIC_TEXTUAL_API_KEY from Colab secrets
os.environ["TONIC_TEXTUAL_API_KEY"] = userdata.get('TONIC_TEXTUAL_API_KEY')

## 1. Document cleaning

**Clean up documents before replacing PII with realistic fake data**

In [13]:
from haystack.dataclasses import Document
from haystack_integrations.components.tonic_textual import TonicTextualDocumentCleaner

cleaner = TonicTextualDocumentCleaner(generator_default="Synthesis")
result = cleaner.run(documents=[
    Document(content="Patient John Smith, DOB 03/15/1982, was admitted for chest pain.")
])
print(result["documents"][0].content)


Patient Alfonzo Uva, DOB 03/20/1982, was admitted for chest pain.


**Tokenize PII**

In [14]:
cleaner = TonicTextualDocumentCleaner(generator_default="Redaction")
result = cleaner.run(documents=[
    Document(content="Contact Jane Doe at jane@example.com.")
])
print(result["documents"][0].content)


Contact [NAME_GIVEN_iKpB3] [NAME_FAMILY_Z3W2] at [EMAIL_ADDRESS_QEgToEQ8FO6hC16fJG].


## Entity Extraction

In [15]:
from haystack.dataclasses import Document
from haystack_integrations.components.tonic_textual import TonicTextualEntityExtractor

extractor = TonicTextualEntityExtractor()
result = extractor.run(documents=[
    Document(content="My name is Afiz Shaik and my email is afiz@example.com.")
])

for entity in TonicTextualEntityExtractor.get_stored_annotations(result["documents"][0]):
    print(f"{entity.entity}: {entity.text} (confidence: {entity.score:.2f})")


NAME_GIVEN: Afiz (confidence: 0.95)
NAME_FAMILY: Shaik (confidence: 0.93)
EMAIL_ADDRESS: afiz@example.com (confidence: 0.99)


## Complete Haystack Pipeline

In [16]:
from haystack import Pipeline
from haystack.dataclasses import Document
from haystack_integrations.components.tonic_textual import (
    TonicTextualDocumentCleaner,
    TonicTextualEntityExtractor,
)

pipeline = Pipeline()
pipeline.add_component("cleaner", TonicTextualDocumentCleaner(generator_default="Synthesis"))
pipeline.add_component("extractor", TonicTextualEntityExtractor())
pipeline.connect("cleaner", "extractor")

result = pipeline.run({
    "cleaner": {
        "documents": [
            Document(content="Contact Jane Doe at jane@example.com or (555) 867-5309."),
        ]
    }
})

for doc in result["extractor"]["documents"]:
    entities = TonicTextualEntityExtractor.get_stored_annotations(doc)
    print(f"Cleaned: {doc.content}")
    print(f"Entities: {[(e.entity, e.text) for e in entities]}")


Cleaned: Contact Antonina Crummitt at pacv@pvjivyj.lfq or (851) 452-9397.
Entities: [('NAME_GIVEN', 'Antonina'), ('NAME_FAMILY', 'Crummitt'), ('EMAIL_ADDRESS', 'pacv@pvjivyj.lfq'), ('PHONE_NUMBER', '(851) 452-9397')]
